# Trace Extraction → Coupling Map

This tutorial walks through the **trace-based spectral extraction** workflow — from a raw flat FITS to a `couplingmap.fits` ready for image reconstruction.

### When to use trace extraction

| Extractor | When to use |
|-----------|-------------|
| `simple_box` | Fixed apertures — traces are nearly straight, quick to set up |
| `trace_box` | Curved traces — follows per-column fiber centers |
| `simple_optimal` | Gaussian-weighted extraction for better S/N |
| `FIRSTPL` | Full 2D optimal (FIRST-PL instrument with `visPLred` model) |

This tutorial covers **`simple_box`** and **`trace_box`** — both driven by a `traces.npz` calibration file.

### Prerequisites

- A flat (or lamp) FITS file — used to locate fiber positions
- A dark FITS file (optional)
- An `average_map.h5` — output of `plred-average` (Step 4)

### Outline

1. [Inspect the flat image](#1-inspect)
2. [Find fiber peaks and traces](#2-traces)
3. [Save `traces.npz`](#3-save)
4. [Write the pipeline config](#4-config)
5. [Run extraction → `couplingmap.fits`](#5-extract)
6. [Inspect the coupling map](#6-inspect)
7. [CLI one-liner reference](#7-cli)

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import h5py
from astropy.io import fits
from configobj import ConfigObj

import PLred.specextract as specextract

## Setup — file paths

Adjust these to match your own data directories.

In [ ]:
# ─── Input files ────────────────────────────────────────────────────────────
FLAT_FITS    = 'data/slowcam/cropped_firstpl_15:05:17.315924371.fits'
DARK_FITS    = 'data/slowcam/dark.fits'
AVERAGE_MAP  = 'timestamp_matching_output/average_map.h5'

# ─── Instrument parameters ──────────────────────────────────────────────────
NFIB = 38       # number of fibers

# ROI used when building average_map.h5 (y0, y1, x0, x1 in detector coords)
# Read directly from the H5 if available:
with h5py.File(AVERAGE_MAP, 'r') as f:
    roi = tuple(int(v) for v in f['metadata/plcam_roi'][:])
    map_n = int(f.attrs['map_n'])
    x_mas = f['metadata/x_mas'][:]
    y_mas = f['metadata/y_mas'][:]

PLCAM_ROI = roi                           # (y0, y1, x0, x1)
XMIN_DET  = roi[2]                        # detector column of first stored pixel
XMAX_DET  = roi[3]

print(f'map_n       = {map_n}')
print(f'plcam_roi   = {PLCAM_ROI}   →  y=[{roi[0]},{roi[1]}), x=[{roi[2]},{roi[3]})')
print(f'nwav stored = {XMAX_DET - XMIN_DET} columns')

# ─── Output paths ───────────────────────────────────────────────────────────
TRACES_NPZ   = 'timestamp_matching_output/traces.npz'
CONFIG_INI   = 'timestamp_matching_output/obs.ini'
OUTPUT_FITS  = 'timestamp_matching_output/coupling_map_trace.fits'

---
<a id='1-inspect'></a>
## 1. Inspect the flat image

Load the flat lamp FITS, dark-subtract, then crop to the same spectral ROI used during averaging.
This ensures the traces are in the same pixel coordinates as the frames stored in `average_map.h5`.

In [ ]:
# Load flat (cube → average)
flat_data = fits.getdata(FLAT_FITS).astype(np.float32)
flat = np.mean(flat_data, axis=0) if flat_data.ndim == 3 else flat_data
print(f'Flat shape (after averaging): {flat.shape}   min={flat.min():.1f}  max={flat.max():.1f}')

# Load dark
dark_data = fits.getdata(DARK_FITS).astype(np.float32)
dark = np.mean(dark_data, axis=0) if dark_data.ndim == 3 else dark_data
print(f'Dark shape: {dark.shape}')

# Dark subtract
flat_ds = flat - dark

# Crop to the ROI used by average_map.h5
y0, y1, x0, x1 = PLCAM_ROI
flat_roi = flat_ds[y0:y1, x0:x1]
print(f'Flat ROI shape (y=[{y0},{y1}), x=[{x0},{x1})): {flat_roi.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: 2D flat image in the ROI
vmin, vmax = np.nanpercentile(flat_roi, [1, 99])
axes[0].imshow(flat_roi, aspect='auto', origin='upper',
               vmin=max(0, vmin), vmax=vmax, cmap='inferno')
axes[0].set_xlabel(f'x (detector column − {x0})')
axes[0].set_ylabel('y (cross-dispersion pixel)')
axes[0].set_title(f'Flat lamp (dark-subtracted, ROI x=[{x0},{x1}))')

# Right: cross-dispersion profile — sum over spectral axis
profile = np.nansum(flat_roi, axis=1)
axes[1].plot(profile, np.arange(len(profile)))
axes[1].invert_yaxis()
axes[1].set_xlabel('Summed counts')
axes[1].set_ylabel('y pixel')
axes[1].set_title('Cross-dispersion profile')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
<a id='2-traces'></a>
## 2. Find fiber peaks and traces

### 2a. Find initial peak positions

`find_peaks` collapses the image along the spectral axis and uses `peakutils` to locate the fibers.  
Tune `thres` (0–1, relative to the maximum) and `min_dist` (pixels) until exactly `NFIB` peaks are found.

In [ ]:
# ▼ Tune these parameters ▼
THRES    = 0.05   # minimum peak height relative to max (0–1)
MIN_DIST = 6      # minimum pixel separation between peaks

ylocs = specextract.find_peaks(flat_roi, nfib=NFIB, thres=THRES, min_dist=MIN_DIST, plot=False)
ylocs = np.sort(ylocs)
print(f'Found {len(ylocs)} peaks at y = {ylocs}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Image with horizontal lines at each fiber
vmin, vmax = np.nanpercentile(flat_roi, [1, 99])
axes[0].imshow(flat_roi, aspect='auto', origin='upper',
               vmin=max(0, vmin), vmax=vmax, cmap='inferno')
for yloc in ylocs:
    axes[0].axhline(yloc, color='cyan', lw=0.8, alpha=0.7)
axes[0].set_title(f'{len(ylocs)} fiber peaks detected')
axes[0].set_xlabel('Spectral channel')
axes[0].set_ylabel('y pixel')

# Cross-dispersion profile with peak markers
profile = np.nansum(flat_roi, axis=1)
axes[1].plot(profile, np.arange(len(profile)), 'k', lw=0.8)
axes[1].plot(profile[ylocs], ylocs, 'ro', ms=6, label='peaks')
axes[1].invert_yaxis()
axes[1].set_xlabel('Summed counts')
axes[1].set_ylabel('y pixel')
axes[1].set_title('Cross-dispersion profile + detected peaks')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('If any peaks look wrong, adjust THRES or MIN_DIST above and re-run.')

### 2b. Trace fiber centers across the spectral axis

`find_traces` follows each fiber column-by-column, fitting a polynomial to smooth out noise.  
For narrow ROIs (few spectral columns), keep `poly_deg` low (1 or 2).

In [ ]:
# ▼ Tune these if traces look wrong ▼
TRACE_WIDTH = 4    # cross-dispersion half-width for tracking
POLY_DEG    = 2    # polynomial degree for trace smoothing

traces = specextract.find_traces(
    flat_roi,
    nfib=NFIB,
    ini_ys=ylocs,
    trace_width=TRACE_WIDTH,
    poly_deg=POLY_DEG,
    plot=False,
)
print(f'traces shape: {traces.shape}   (nfib={NFIB}, nwav={flat_roi.shape[1]})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Image with overlaid traces
vmin, vmax = np.nanpercentile(flat_roi, [1, 99])
axes[0].imshow(flat_roi, aspect='auto', origin='upper',
               vmin=max(0, vmin), vmax=vmax, cmap='inferno')
x_arr = np.arange(flat_roi.shape[1])
for fi, trace in enumerate(traces):
    axes[0].plot(x_arr, trace, color=f'C{fi % 10}', lw=0.8, alpha=0.8)
axes[0].set_title('Flat image + fiber traces')
axes[0].set_xlabel('Spectral channel')
axes[0].set_ylabel('y pixel')

# Trace curvature — deviation from initial y-position
for fi, trace in enumerate(traces):
    axes[1].plot(x_arr, trace - trace[flat_roi.shape[1]//2],
                 color=f'C{fi % 10}', lw=0.8, alpha=0.8)
axes[1].axhline(0, color='k', lw=0.5, ls='--')
axes[1].set_xlabel('Spectral channel')
axes[1].set_ylabel('Δy from midpoint (pixels)')
axes[1].set_title('Trace curvature per fiber')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

max_curve = float(np.nanmax(np.abs(traces - traces[:, [flat_roi.shape[1]//2]])))
print(f'Max trace curvature: {max_curve:.2f} pixels')
if max_curve < 0.5:
    print('→ Traces are nearly straight — simple_box extractor is fine.')
else:
    print('→ Noticeable curvature — trace_box extractor recommended.')

---
<a id='3-save'></a>
## 3. Save `traces.npz`

`make_trace_file` repeats steps 2a–2b automatically from a FITS path and writes a self-contained `traces.npz`.  
Use this function (or the `plred-traces` CLI) in real workflows rather than saving arrays manually.

In [ ]:
specextract.make_trace_file(
    fits_path   = FLAT_FITS,
    nfib        = NFIB,
    dark_path   = DARK_FITS,
    xmin        = XMIN_DET,     # crop to the same spectral range as the H5
    xmax        = XMAX_DET,
    thres       = THRES,
    min_dist    = MIN_DIST,
    trace_width = TRACE_WIDTH,
    poly_deg    = POLY_DEG,
    outpath     = TRACES_NPZ,
    plot        = False,
    verbose     = True,
)
print(f'\nSaved: {TRACES_NPZ}')

In [ ]:
# Verify the saved file
d = np.load(TRACES_NPZ, allow_pickle=False)
print('Keys:', list(d.keys()))
print(f'  ylocs  : {d["ylocs"].shape}  → {d["ylocs"]}')
print(f'  traces : {d["traces"].shape}  (nfib, nwav)')
print(f'  nfib   : {int(d["nfib"])}')
print(f'  xmin   : {int(d["xmin"])}  xmax : {int(d["xmax"])}  (detector coords)')

---
<a id='4-config'></a>
## 4. Write the pipeline config

The unified config drives `plred-extract` (and all other CLIs).  
Here we only need the `[Specextract]` section — other sections are left blank and ignored.

In [ ]:
# Choose extractor type:
#   'simple_box'  — fixed aperture, fast, good for straight traces
#   'trace_box'   — follows curved traces, better for bent fibers
EXTRACTOR = 'trace_box'

cfg = ConfigObj()
cfg.filename = CONFIG_INI

cfg['Instrument'] = {
    'nfib'                : str(NFIB),
    'spectral_orientation': 'horizontal',
}

cfg['Specextract'] = {
    'input'     : AVERAGE_MAP,
    'extractor' : EXTRACTOR,
    'output'    : OUTPUT_FITS,
    'plcam_roi' : ','.join(str(v) for v in PLCAM_ROI),
    'trace_file': TRACES_NPZ,
    # unused for box extractors:
    'profile_file': '',
    'model_file'  : '',
    'nonlin_file' : '',
}

cfg.write()
print(f'Config written to: {CONFIG_INI}')
print()

# Print config for inspection
with open(CONFIG_INI) as fh:
    print(fh.read())

---
<a id='5-extract'></a>
## 5. Run extraction → `couplingmap.fits`

### Option A — Python API

In [ ]:
specextract.extract_from_config(CONFIG_INI)
print(f'\nOutput: {OUTPUT_FITS}')

### Option B — CLI

Equivalent one-liner from the terminal:

```bash
plred-extract timestamp_matching_output/obs.ini
```

---
<a id='6-inspect'></a>
## 6. Inspect the coupling map

In [ ]:
from astropy.io import fits as pyfits

with pyfits.open(OUTPUT_FITS) as hdul:
    hdul.info()
    print()
    hdr  = hdul[0].header
    data = hdul[0].data     # (map_n, map_n, nfib, nwav)
    nfr  = hdul[1].data     # (map_n, map_n)  — frames per bin
    var  = hdul[3].data     # variance

print(f'Coupling map shape : {data.shape}  (map_n={map_n}, nfib={NFIB}, nwav={data.shape[3]})')
print(f'Extractor type     : {hdr.get("HIERARCH SPEX TYPE", "n/a")}')
print(f'Frames per bin:\n{nfr}')

valid_bins = int(np.sum(nfr >= 1))
print(f'Valid bins         : {valid_bins} / {map_n * map_n}')

In [ ]:
# Mean coupling map per fiber (average over spectral channels)
mean_map = np.nanmean(data, axis=3)   # (map_n, map_n, nfib)

ncols = 8
nrows = int(np.ceil(NFIB / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))
axes = axes.flatten()

extent = [x_mas[0], x_mas[-1], y_mas[-1], y_mas[0]]

for fi in range(NFIB):
    ax = axes[fi]
    m    = mean_map[:, :, fi]
    vmax = np.nanpercentile(m[m > 0], 98) if np.any(m > 0) else 1.0
    ax.imshow(m, extent=extent, origin='upper',
              cmap='inferno', vmin=0, vmax=max(vmax, 1e-9), aspect='equal')
    ax.set_title(f'fib {fi}', fontsize=7)
    ax.tick_params(labelsize=5)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[NFIB:]:
    ax.set_visible(False)

fig.suptitle(f'Mean coupling maps — {EXTRACTOR} extractor', y=1.01)
fig.text(0.5, -0.01, 'x (mas)', ha='center')
fig.text(-0.01, 0.5, 'y (mas)', va='center', rotation='vertical')
plt.tight_layout()
plt.show()

In [ ]:
# Spectra of the most-populated bin
ix, iy = np.unravel_index(np.argmax(nfr), nfr.shape)
spec = data[ix, iy]   # (nfib, nwav)

fig, ax = plt.subplots(figsize=(9, 3))
for fi in range(NFIB):
    ax.plot(spec[fi], color=f'C{fi % 10}', lw=0.8, alpha=0.7)
ax.set_xlabel('Spectral channel (pixel)')
ax.set_ylabel('Counts')
ax.set_title(f'Extracted spectra — bin ({ix},{iy}),  nframes={nfr[ix,iy]}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Optional: compare simple_box vs trace_box ────────────────────────────────
#
# Repeat with the other extractor and overlay spectra for one fiber.

other_extractor = 'simple_box' if EXTRACTOR == 'trace_box' else 'trace_box'
other_output    = OUTPUT_FITS.replace('.fits', f'_{other_extractor}.fits')

cfg2 = ConfigObj(CONFIG_INI)
cfg2['Specextract']['extractor'] = other_extractor
cfg2['Specextract']['output']    = other_output
cfg2.write()
specextract.extract_from_config(CONFIG_INI)

# Restore original config
cfg2['Specextract']['extractor'] = EXTRACTOR
cfg2['Specextract']['output']    = OUTPUT_FITS
cfg2.write()

with pyfits.open(other_output) as hdul:
    data2 = hdul[0].data

fib_i = 0
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(spec[fib_i],           label=EXTRACTOR,       lw=1.5)
ax.plot(data2[ix, iy, fib_i], label=other_extractor, lw=1.0, ls='--', alpha=0.8)
ax.set_xlabel('Spectral channel')
ax.set_ylabel('Counts')
ax.set_title(f'Fiber {fib_i}  —  {EXTRACTOR} vs {other_extractor}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id='7-cli'></a>
## 7. CLI one-liner reference

The cells above show the Python API.  The same workflow runs entirely from the terminal.

### Step A — Generate `traces.npz` (once per instrument config)

```bash
plred-traces data/slowcam/cropped_firstpl_15:05:17.315924371.fits \
    --dark data/slowcam/dark.fits \
    --nfib 38 \
    --xmin 1200 --xmax 1220 \
    --thres 0.05 --min-dist 6 \
    --out timestamp_matching_output/traces.npz
```

Remove `--no-plot` (or omit it) to see the interactive diagnostic plot.

### Step B — Run extraction (per observation)

```bash
plred-extract timestamp_matching_output/obs.ini
```

### Step C — Full pipeline run

If you want to re-run from averaging onwards:

```bash
plred-run timestamp_matching_output/obs.ini --from 4
```

Or run individual steps:

```bash
plred-average timestamp_matching_output/obs.ini   # step 4 → map.h5
plred-extract timestamp_matching_output/obs.ini   # step 5 → couplingmap.fits
```

---

**Next step:** Load the coupling map into `CouplingMapModel` and run image reconstruction.

```python
from PLred.mapmodel import CouplingMapModel
model = CouplingMapModel(mapdata='timestamp_matching_output/coupling_map_trace.fits', min_nframes=2)
```

See **[Step 3: Image reconstruction](step3_image_reconstruction.ipynb)** for the full reconstruction workflow.